## verifier_constat_par_type

**Fichier(s) source :** `./data/bofip_stock_live_20260521.tgz` (stock du 21.05.2026) et `./data/inventaire_bofip_stock_live_20260521.xlsx` (inventaire produit par `profilage_bofip_consolide.ipynb`)

**Fichier(s) de sortie :** `./data/mesure_par_document_types_mineurs.csv` et `./data/synthese_par_type.csv`

**Description :** Transforme l'illustration par exemple en mesure sur l'ensemble : pour les cinq types mineurs (hors Commentaire, environ 632 documents), parcourt tous les documents dans l'archive et compte des indicateurs de structure (documents sans texte, jetons de gabarit non résolus, coquilles, présence de tableaux et d'images, longueur médiane), avec une synthèse par type.

## Etape 1. Chemins

In [1]:
import os, glob, re, tarfile, pandas as pd

TGZ = r"./data/bofip_stock_live_20260521.tgz"
INVENTAIRE = r"./data/inventaire_bofip_stock_live_20260521.xlsx"
BASE = r"./data"
def _ch(motif):
    return glob.glob(os.path.join(BASE,'**',motif), recursive=True) if os.path.isdir(BASE) else []
if not os.path.exists(TGZ):
    t=_ch('*stock*live*.tgz'); TGZ=t[0] if t else TGZ
if not os.path.exists(INVENTAIRE):
    t=_ch('inventaire*stock*.xlsx'); INVENTAIRE=t[0] if t else INVENTAIRE
print('Archive    :', os.path.exists(TGZ), TGZ)
print('Inventaire :', os.path.exists(INVENTAIRE), INVENTAIRE)

Archive    : True ./data/bofip_stock_live_20260521.tgz
Inventaire : True ./data/inventaire_bofip_stock_live_20260521.xlsx


## Etape 2. Lister les identifiants des types mineurs

On vise les cinq types mineurs (hors Commentaire), soit environ 632 documents.

In [2]:
df = pd.read_excel(INVENTAIRE, dtype={'identifiant': str})
TYPES_MINEURS = [t for t in df['type'].dropna().unique() if str(t) != 'Commentaire']
id_type = {}
for _, r in df[df['type'].isin(TYPES_MINEURS)].iterrows():
    id_type[str(r['identifiant'])] = str(r['type'])
print('Types mineurs :', TYPES_MINEURS)
print('Documents a examiner :', len(id_type))

Types mineurs : ['Lettre Type / Modèle', 'Formulaire', 'Barème', 'Autres annexes', 'Cartographie']
Documents a examiner : 632


## Etape 3. Lire le contenu de chaque document et mesurer

On parcourt l'archive une seule fois. Pour chaque document de type mineur, on lit son data.html et on calcule les indicateurs.

In [3]:
def sans_balises(html):
    txt = re.sub(r'<[^>]+>', ' ', html)
    return re.sub(r'\s+', ' ', txt).strip()

lignes = []
with tarfile.open(TGZ, 'r:gz') as tar:
    for m in tar:
        if not m.isfile() or not m.name.endswith('data.html'): continue
        mobj = re.search(r'/(\d+-PGP)/', m.name.replace('\\','/'))
        if not mobj: continue
        ident = mobj.group(1)
        if ident not in id_type: continue
        html = tar.extractfile(m).read().decode('utf-8', errors='replace')
        corps = html.split('<body>')[-1]
        texte = sans_balises(corps)
        lignes.append({
            'identifiant': ident,
            'type': id_type[ident],
            'longueur_texte': len(texte),
            'nb_images': len(re.findall(r'<img', corps)),
            'nb_tableaux': len(re.findall(r'<table', corps)),
            'jeton_casse': bool(re.search(r'\[node:|\[\w+:\w+\]|\$field|\{\{', corps)),
            'coquille': bool(re.search(r'sont retir|transf[ée]r[ée]|compter de la date de publication', texte, re.I)),
            'sans_texte': len(texte) < 50,
        })
mes = pd.DataFrame(lignes)
print('Documents lus :', len(mes))
mes.head()

Documents lus : 632


,identifiant,type,longueur_texte,nb_images,nb_tableaux,jeton_casse,coquille,sans_texte
0,1004-PGP,Lettre Type / Modèle,472,0,0,False,False,False
1,1006-PGP,Lettre Type / Modèle,1478,0,5,False,False,False
2,1135-PGP,Lettre Type / Modèle,1524,0,2,False,False,False
3,1446-PGP,Lettre Type / Modèle,878,0,0,True,True,False
4,1796-PGP,Lettre Type / Modèle,878,0,0,True,True,False


## Etape 4. Synthese par type

On agrege les indicateurs par type : combien de documents sans texte, combien avec un jeton casse, combien de coquilles, combien avec tableau.

In [4]:
synth = mes.groupby('type').agg(
    documents=('identifiant','count'),
    sans_texte=('sans_texte','sum'),
    avec_image=('nb_images', lambda s: (s>0).sum()),
    avec_tableau=('nb_tableaux', lambda s: (s>0).sum()),
    jeton_casse=('jeton_casse','sum'),
    coquilles=('coquille','sum'),
    longueur_mediane=('longueur_texte','median'),
).reset_index()
pd.set_option('display.width', 160)
print(synth.to_string(index=False))

                type  documents  sans_texte  avec_image  avec_tableau  jeton_casse  coquilles  longueur_mediane
      Autres annexes        306          27          43           131           10         87            1190.0
              Barème         36           0           6            21            1          6            1963.5
        Cartographie          5           4           5             0            0          0               0.0
          Formulaire         84          12          15            52            1          9            1098.5
Lettre Type / Modèle        201           9          12            52           10         23            1478.0


## Etape 5. Enregistrer les resultats

Le detail par document et la synthese par type sont enregistres a cote de l'archive.

In [5]:
dossier = os.path.dirname(TGZ)
mes.to_csv(os.path.join(dossier,'mesure_par_document_types_mineurs.csv'), sep=';', index=False, encoding='utf-8-sig')
synth.to_csv(os.path.join(dossier,'synthese_par_type.csv'), sep=';', index=False, encoding='utf-8-sig')
print('Enregistres dans', dossier, ':')
print('   mesure_par_document_types_mineurs.csv')
print('   synthese_par_type.csv')

Enregistres dans ./data :
   mesure_par_document_types_mineurs.csv
   synthese_par_type.csv
